In [3]:
!nvidia-smi

Thu Apr 16 08:04:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
%%writefile shared_matmul.cu
#include<stdio.h>

#define TILE_WIDTH 16

// shared matrix multiplication
__global__ void shared_mm(float *A, float *B, float *C, int N){
  // first we load thread and block wise details (basically matrix A,B)
  int bx = blockIdx.x;
  int by = blockIdx.y;
  int tx = threadIdx.x;
  int ty = threadIdx.y;

  // now we load Matrix C that's for output
  // i = row index ; j = tile number * tile width + thread index
  int i = blockDim.y*by + ty;
  int j = blockDim.x*bx + tx;

  // now we allocate shared memory by breaking into equal dimension matrices
  __shared__ float sh_A[TILE_WIDTH][TILE_WIDTH];
  __shared__ float sh_B[TILE_WIDTH][TILE_WIDTH];

  float x = 0.0;

  for (int tile_num = 0; tile_num < N/TILE_WIDTH; tile_num++){
    sh_A[ty][tx] = A[(i)*N + tile_num*TILE_WIDTH + tx];
    sh_B[ty][tx] = B[(tile_num*TILE_WIDTH + ty)*N + j];
    __syncthreads();  // use sync to stop exec until all threads have reached a point where all tiles are loaded
    // since the finishing time for the above operation may be different, we use sync

    // now we do dot product which is pretty trivial
    for(int t = 0; t < TILE_WIDTH; t++){
      x += sh_A[ty][t]*sh_B[t][tx];
    }
    __syncthreads();  // again make sure all threads have completed their exec before moving on to next tile
  }

  // now store the result in mtx C (ONCE, after all tiles)
  C[i*N+j] = x;
}
int main(){
  int n = 512;
  size_t bytes = n*n*sizeof(float);
  float *h_A, *h_B, *h_C;
  h_A = (float*)malloc(bytes);
  h_B = (float*)malloc(bytes);
  h_C = (float*)malloc(bytes);
  for(int i =0; i<n*n;i++){
      h_A[i]= (float)rand()/RAND_MAX;
      h_B[i]= (float)rand()/RAND_MAX;
  }
  float *d_A, *d_B, *d_C;
  cudaMalloc(&d_A,bytes);
  cudaMalloc(&d_B,bytes);
  cudaMalloc(&d_C,bytes);
  cudaMemcpy(d_A,h_A,bytes,cudaMemcpyHostToDevice);
  cudaMemcpy(d_B,h_B,bytes,cudaMemcpyHostToDevice);
  dim3 threadsPerBlock(TILE_WIDTH, TILE_WIDTH);
  dim3 numBlocks(n/TILE_WIDTH, n/TILE_WIDTH);
  printf("Launching kernel, blocks = %dx%d ; threads per block %dx%d \n",
         numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);
  shared_mm<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, n);
  cudaDeviceSynchronize();
   // Timing
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);
  cudaEventRecord(start);
  shared_mm<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C, n);
  cudaEventRecord(stop);
  cudaEventSynchronize(stop);
  float gpu_time = 0;
  cudaEventElapsedTime(&gpu_time, start, stop);
  // Copy result back
  cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
  printf("GPU execution time: %.3f ms\n", gpu_time);

  // Print a small section of the resulting matrix C
  printf("\nResulting matrix C (top-left 5x5 sub-matrix):\n");
  for (int i = 0; i < 5; ++i) {
    for (int j = 0; j < 5; ++j) {
      printf("%.2f ", h_C[i * n + j]);
    }
    printf("\n");
  }

  cudaFree(d_A);cudaFree(d_B);cudaFree(d_C);
  free(h_A);free(h_B);free(h_C);
  return 0;
}

Overwriting shared_matmul.cu


In [13]:
!nvcc shared_matmul.cu -o shared_matmul && ./shared_matmul


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Launching kernel, blocks = 32x32 ; threads per block 16x16 
GPU execution time: 0.647 ms

Resulting matrix C (top-left 5x5 sub-matrix):
133.29 136.72 136.92 131.18 129.23 
130.69 130.55 130.02 128.32 121.42 
128.91 133.68 129.08 126.24 123.07 
132.15 131.88 133.18 124.68 122.53 
126.94 128.28 123.16 123.24 122.37 


In [16]:
%%writefile matmul.cu
#include<stdio.h>

//matrix multiplication
__global__ void  matMul(float* A, float* B, float* C, int n){
    int row = blockIdx.y*blockDim.y + threadIdx.y;
    int col = blockIdx.x*blockDim.x + threadIdx.x;
    float sum =0.0f;
    for(int i =0; i<n; i++){ // Changed 1024.0 to n for consistency with matrix size
        sum += A[row*n+i]*B[i*n + col];  // row start hoti hai from index row*n and then go i steps ahead
    }
    C[row*n+col]=sum;
}
int main(){
  int n = 1024; // Matrix dimension
  size_t size = n*n*sizeof(float);
  dim3 threadsPerBlock(16,16); // 16x16 = 256 threads per block but in 2d
  dim3 blocksPergrid(n/16,n/16); // sufficient to cover an nxn mtx

  // Host memory allocation
  float *ha = (float*)malloc(size);
  float *hb = (float*)malloc(size);
  float *hc = (float*)malloc(size);

  // Initialize host matrices
  for(int i=0; i< n;i++){
    for(int j =0; j<n;j++){
      ha[i*n+j] = (float)rand()/RAND_MAX; // Initialize with random values for consistency
      hb[i*n+j] = (float)rand()/RAND_MAX; // Initialize with random values for consistency
    }
  }

  // Device memory allocation
  float *da,*db,*dc;
  cudaMalloc(&da,size);
  cudaMalloc(&db,size);
  cudaMalloc(&dc,size);

  // Copy host to device
  cudaMemcpy(da,ha,size,cudaMemcpyHostToDevice);
  cudaMemcpy(db,hb,size,cudaMemcpyHostToDevice);

  printf("Launching kernel (Standard MM), blocks = %dx%d ; threads per block %dx%d \n",
         blocksPergrid.x, blocksPergrid.y, threadsPerBlock.x, threadsPerBlock.y);

  // First launch to warm up CUDA
  matMul<<<blocksPergrid,threadsPerBlock>>>(da,db,dc,n);
  cudaError_t err = cudaGetLastError();
  if(err != cudaSuccess)
    printf("Kernel error: %s\n", cudaGetErrorString(err));
  cudaDeviceSynchronize();

  // Timing
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);
  cudaEventRecord(start);

  // Actual timed kernel launch
  matMul<<<blocksPergrid,threadsPerBlock>>>(da,db,dc,n);
  cudaEventRecord(stop);
  cudaEventSynchronize(stop);

  float gpu_time = 0;
  cudaEventElapsedTime(&gpu_time, start, stop);
  printf("GPU execution time (Standard MM): %.3f ms\n", gpu_time);

  // Copy result back to host
  cudaMemcpy(hc,dc,size,cudaMemcpyDeviceToHost);

  // Print a small section of the resulting matrix C
  printf("\nResulting matrix C (top-left 5x5 sub-matrix) from Standard MM:\n");
  for (int i = 0; i < 5; ++i) {
    for (int j = 0; j < 5; ++j) {
      printf("%.2f ", hc[i * n + j]);
    }
    printf("\n");
  }

  // Free device memory
  cudaFree(da);cudaFree(db);cudaFree(dc);
  // Free host memory
  free(ha);free(hb);free(hc);

  return 0;
}

Overwriting matmul.cu


In [17]:
!nvcc matmul.cu -o matmul && ./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Launching kernel (Standard MM), blocks = 64x64 ; threads per block 16x16 
GPU execution time (Standard MM): 9.192 ms

Resulting matrix C (top-left 5x5 sub-matrix) from Standard MM:
257.26 261.44 266.63 255.50 260.80 
253.51 262.81 260.48 255.07 257.75 
253.81 257.39 263.54 249.16 252.88 
257.95 267.55 273.37 261.92 263.21 
249.62 251.48 256.19 248.08 245.03 
